This code is building a complete mini RAG (Retrieval-Augmented Generation) system using:

Documents → Embeddings → Chroma Vector Store → Retriever → Prompt → LLM

                  YOUR DOCUMENTS
                        │
                        ▼
              HuggingFace Embeddings
                        │
                  convert text
                  into vectors
                        │
                        ▼
                Chroma Vector Store
                        │
                 similarity search
                        │
                        ▼
                   Retriever
                        │
                 relevant documents
                        │
                        ▼
                    RAG Prompt
                        │
                  Context + Question
                        │
                        ▼
                       LLM
                        │
                        ▼
                     Answer

### Vector stores and retrievers
This video tutorial will familiarize you with LangChain's vector store and retriever abstractions. These abstractions are designed to support retrieval of data-- from (vector) databases and other sources-- for integration with LLM workflows. They are important for applications that fetch data to be reasoned over as part of model inference, as in the case of retrieval-augmented generation.

We will cover 
- Documents
- Vector stores
- Retrievers


### Documents
LangChain implements a Document abstraction, which is intended to represent a unit of text and associated metadata. It has two attributes:

- page_content: a string representing the content;
- metadata: a dict containing arbitrary metadata.
The metadata attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual Document object often represents a chunk of a larger document.

Let's generate some sample documents:

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [3]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(groq_api_key=groq_api_key,model="openai/gpt-oss-20b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x3015ef4d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x30165c190>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5083.17it/s]


In [16]:
## VectorStores
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(documents,embedding=embeddings)
vectorstore


In [17]:
vectorstore.similarity_search("cat")

[Document(id='03d6f8b4-e166-4ce2-9647-b63688d77219', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='5c378631-26ca-41cc-b3d1-7ac6ce810e2a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='6ba3390d-f383-48fd-8bd8-a8deacd18c5f', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='a3c9fe6f-c9d3-4867-99a9-1a21394b82f5', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [18]:
## Async query
await vectorstore.asimilarity_search("cat")

[Document(id='03d6f8b4-e166-4ce2-9647-b63688d77219', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='5c378631-26ca-41cc-b3d1-7ac6ce810e2a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='6ba3390d-f383-48fd-8bd8-a8deacd18c5f', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='a3c9fe6f-c9d3-4867-99a9-1a21394b82f5', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [19]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='03d6f8b4-e166-4ce2-9647-b63688d77219', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056814193726),
 (Document(id='5c378631-26ca-41cc-b3d1-7ac6ce810e2a', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056814193726),
 (Document(id='6ba3390d-f383-48fd-8bd8-a8deacd18c5f', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740900039672852),
 (Document(id='a3c9fe6f-c9d3-4867-99a9-1a21394b82f5', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.5740900039672852)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

We can create a simple version of this ourselves, without subclassing Retriever. If we choose what method we wish to use to retrieve documents, we can create a runnable easily. Below we will build one around the similarity_search method:

In [24]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='03d6f8b4-e166-4ce2-9647-b63688d77219', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='6ba3390d-f383-48fd-8bd8-a8deacd18c5f', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

Vectorstores implement an as_retriever method that will generate a Retriever, specifically a VectorStoreRetriever. These retrievers include specific search_type and search_kwargs attributes that identify what methods of the underlying vector store to call, and how to parameterize them. For instance, we can replicate the above with the following:

In [21]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])


[[Document(id='03d6f8b4-e166-4ce2-9647-b63688d77219', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='6ba3390d-f383-48fd-8bd8-a8deacd18c5f', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [ ]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm    ##Send the question to the retriever to get context, keep the original question, put both into the prompt, and send the result to the LLM.

response=rag_chain.invoke("tell me about dogs")
print(response.content)


Dogs are great companions, known for their loyalty and friendliness.


                     USER QUESTION
                           │
                           │
                 "tell me about dogs"
                           │
             ┌─────────────┴─────────────┐
             │                           │
             ▼                           ▼
         retriever               RunnablePassthrough
             │                           │
             ▼                           ▼
       Chroma search              original question
             │                           │
             ▼                           │
      Relevant document                 │
             │                           │
             └─────────────┬─────────────┘
                           │
                           ▼
                  {
                    context: document,
                    question: question
                  }
                           │
                           ▼
                    ChatPromptTemplate
                           │
                           ▼
              "Answer using context only..."
                           │
                           ▼
                          LLM
                           │
                           ▼
                       AIMessage
                           │
                           ▼
                    response.content

a real RAG pipeline is more like:

PDF / Website / Database
          ↓
      Documents
          ↓
     Text Splitter
          ↓
       Chunks
          ↓
   Embedding Model
          ↓
       Vectors
          ↓
    Vector Database
          ↓
       Retriever
          ↓
 Relevant Chunks
          ↓
      LLM Prompt

LangChain is mostly orchestrating the process.

The actual language understanding/generation happens inside the LLM.

code has several different components:

LangChain
   │
   ├── Prompt construction
   ├── Retriever
   ├── Vector database
   └── LLM interface
             │
             ▼
           LLM
             │
             ▼
        Transformer

How does the LLM answer?

Prompt
  ↓
Tokenization
  ↓
Token embeddings
  ↓
Positional information
  ↓
Transformer layers
  ↓
Self-attention
  ↓
Predict next token
  ↓
Predict next token
  ↓
Predict next token
  ↓
Final answer